In [ ]:
import json
import random
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Константы
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'

### Читаем данные

In [ ]:
# Читаем размеченные данные
data = pd.read_csv('data/marked_data.csv')
data.head(5)

In [ ]:
# Распределение размеченных данных
print(data['category'].value_counts())

### Анализ ошибок и корректировка

In [ ]:
# Печать данных определенной категории
def print_category(category):
    for text in data[data['category'] == category]['text'].to_list():
        print(text)
        print()

In [ ]:
# После автоматической разметки в категорию <бытовая техника> попали все отзывы со словом <стирка> и его формы
print_category('бытовая техника')

In [ ]:
# После автоматической разметки в категорию <посуда> попали все отзывы про бюстгалтеры, из-за слова <чашка>
print_category('посуда')

In [ ]:
# Явно неправильная разметка
print_category('электроника')

In [ ]:
# Новое распределение данных
data = data[~data['category'].isin(['бытовая техника', 'электроника', 'нет категории'])]
data.loc[data['category'] == 'посуда', 'category'] = 'одежда'
print(data['category'].value_counts())

### Генерация синтетических данных

In [ ]:
# Загружаем модель и токенизатор
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype='auto', device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Категории для которых надо генерировать данные
categories = ['бытовая техника', 'обувь', 'посуда', 'текстиль', 'товары для детей', 'украшения и аксессуары', 'электроника']

# Категории товаров с описанием
description = {
    'бытовая техника': 'холодильники, стиральные машины, плиты, микроволновки, чайники, пылесосы и другая техника для дома.',
    'обувь': 'кроссовки, ботинки, туфли, сандалии, сапоги и другая обувь для взрослых и детей.',
    'посуда': 'только тарелки, кружки, чашки для еды, столовые приборы и кухонная утварь; НЕ включает одежду, бельё или чашки бюстгальтеров.',
    'текстиль': 'только постельное бельё, покрывала, наволочки, кухонные полотенца, пледы, портьеры, шторы, ковры и так далее.',
    'товары для детей': 'игрушки, детская одежда, детская мебель, товары для ухода за детьми, коляски, автокресла.',
    'украшения и аксессуары': 'серьги, кольца, браслеты, ожерелья, ремни, шарфы, сумки, очки и другие модные аксессуары.',
    'электроника': 'смартфоны, планшеты, ноутбуки, камеры, наушники, колонки, смарт-часы и прочая электроника.',
}

# Категории товаров с примерами
examples = {
    'бытовая техника': [
        'Заказала стиральную машину, пришла быстро, упаковка целая, но при включении гремит и сильно прыгает по полу.',
        'Пришёл с небольшой вмятиной, но работает без проблем. Доставка быстрая.',
        'Плита греет хорошо, но запах пластика не проходил неделю. Сейчас всё нормально.'
    ],
    'обувь': [
        'Купила новые кроссовки, очень удобные.',
        'Качество отличное, запаха нет, фурнитура надёжная. Брала мужу — доволен.',
        'Выглядят стильно, но ремешок короткий — на широкую ногу не подойдут.'
    ],
    'посуда': [
        'Тарелки пришли целые, хорошо упакованы. Очень красивые, рисунок яркий. Мыть можно в посудомойке — ничего не стерлось.',
        'Контейнеры хорошие, крышки плотно закрываются. Удобно брать еду с собой на работу.',
        'Ножи тупые, пришлось точить сразу. За эту цену ожидала лучшее качество.'
    ],
    'текстиль': [
        'За свою цену просто находка. Мягкие, но при этом плотные. Взяла набор, теперь думаю заказать ещё.',
        'Цвет вживую немного теплее, чем на фото, но в интерьере даже лучше смотрится. Материал качественный, не просвечивает.',
        'Купила постельное бельё из хлопка, качество хорошее.'
    ],
    'товары для детей': [
        'Игрушка пришла быстро, без запаха, ребёнок в восторге.',
        'Размер не соответсвует ,немного не тот цвет,рассчитан для девочек лет 10-12',
        'Тонкая синтетика, но за такие деньги норм, ребенок доволен'
    ],
    'украшения и аксессуары': [
        'у очков в белой оправе было деформировано стекло , все как будто "плывет"',
        'продавец слукавил. материал шарфв не является обещанным кашемиром.',
        'Браслет выглядит дорого, но застёжка слабая — расстегнулся через день, пришлось подклеить'
    ],
    'электроника': [
        'Наушники ужасные, звук глухой и один перестал работать через неделю.',
        'Клавиатура удобная, экран яркий, звук чистый. Всё как в описании.',
        'Кабель прочный, телефон заряжается быстро. Закажу ещё несколько.'
    ]
}

# Стили для генерации разнообразных отзывов
styles = [
    'Сделай отзывы радостными, покупатель доволен, эмоции счастья и удовлетворения от товара',
    'Сделай отзывы критическими, покупатель разочарован, укажи недостатки товара или проблемы с размером, качеством, доставкой',
    'Отзывы нейтральные, описательные, без сильной эмоции, упор на детали товара, доставки и упаковки',
    'Добавь упоминание упаковки, цвета, размера, сроков доставки',
    'Очень краткие отзывы, 1–2 предложения, просто и понятно'
]

# Дополнительная информация для разметки
meta_info = '''Отзывы в стиле маркетплейса, 1–3 коротких предложения, допустимы лёгкие орфографические ошибки.
Старайся, чтобы каждый отзыв отличался формулировками, упоминай разные детали: запах, внешний вид, 
удобство, упаковку, срок службы, соответствие фото, цену.
'''

# Собираем полную информацию для категорий
full_info = {}
for cat in categories:
    full_info[cat] = {'desc': description[cat], 'examples': examples[cat]}

In [ ]:
# Функция для составления промпта для генерации
def generation_prompt_builder(category_name, category_description, num_examples, seed_examples, style, meta_info=None):
    prompt = f'''
Сгенерируй {num_examples} реалистичных отзывов на русском языке для категории: {category_name}.
Описание категории: {category_description}
{meta_info if meta_info else ''}
{style}\n
'''

    examples = '\n'.join([f'{i + 1}. {ex}' for i, ex in enumerate(seed_examples)])
    prompt += 'Вот примеры реальных отзывов:\n'
    prompt += examples + '\n'

    prompt += '\nОтвет верни строго в формате JSON-массива строк, например:\n["отзыв_1", "отзыв_2"]'

    return prompt

# Получение ответа от llm
def get_llm_answer(prompt, max_tokens=512, temperature=0.0, top_p=1.0, do_sample=False):
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    return response

In [ ]:
# Генерация данных
BATCH_SIZE = 5
result = {}
for cat in categories:
    result[cat] = set()

for cat in full_info.keys():
    for i in tqdm(range(60), desc=f'Генерация для категории: {cat}', total=60):
        try:
            answer = get_llm_answer(generation_prompt_builder(
                cat,
                full_info[cat]['desc'],
                BATCH_SIZE,
                full_info[cat]['examples'],
                random.choice(styles)
            ), 4096, 0.7, 0.9, True)
            result[cat].update(json.loads(answer))
        except Exception as e:
            print(f'Что то пошло не так: {e}')

### Генерация данных для проблемных ситуаций

In [ ]:
# Промпт для отзывов на бюстгальтеры
bra_prompt = f'''
Сгенерируй {BATCH_SIZE} реалистичных отзывов на русском языке о женских бюстгалтерах.
Важно: в каждом отзыве используй слово <чашка> или близкие по смыслу выражения 
(чашечки, чашечка хорошо держит форму, чашка мала/велика, чашка плотная) именно в контексте нижнего белья, а не посуды.
{meta_info}
'''
bra_prompt += '\nОтвет верни строго в формате JSON-массива строк, например:\n["отзыв_1", "отзыв_2"]'

# Промпт для отзывов на одежду
wash_prompt = f'''
Сгенерируй {BATCH_SIZE} реалистичных отзывов на русском языке о товарах категории <одежда>
(платья, брюки, юбки, футболки, свитеры, куртки, бельё, бюстгальтеры, верхняя одежда).
В каждом отзыве обязательно упомяни слово <стирка> или выражения, связанные со стиркой одежды: 
после стирки, не полиняла, не села, пережила несколько стирок.
Важно: слово <стирка> должно относиться только к уходу за одеждой, а не к бытовой технике.
{meta_info}
'''
wash_prompt += '\nОтвет верни строго в формате JSON-массива строк, например:\n["отзыв_1", "отзыв_2"]'

In [ ]:
# Генерация данных
for prompt in (bra_prompt, wash_prompt):
    for i in tqdm(range(20), total=20):
        try:
            prompt += random.choice(styles)
            answer = get_llm_answer(prompt, 4096, 0.7, 0.9, True)
            result['одежда'].update(json.loads(answer))
        except Exception as e:
            print(f'Что то пошло не так: {e}')  

### Сохранение результата

In [ ]:
# Перевод в корректный формат
generated_data = []
for category, reviews in result.items():
    for review in reviews:
        generated_data.append({'category': category, 'review': review})

In [ ]:
generated_data = pd.DataFrame(generated_data)
generated_data.to_csv('data/generated_data.csv', index=False)